In [1]:
import os
import pandas as pd
import re
from datetime import timedelta

# this function reads the .xlsx files from results directory, collect the training time in epochs
# and calculate the mean
def calculate_training_epochs_mean_per_dataset(network_retults_path, network_file_prefix=None):


    xlsx_files = [f for f in os.listdir(network_retults_path) if f.endswith(".xlsx")]

    
    if network_file_prefix is not None:
        #xlsx_files = [f for f in xlsx_files if f.startswith(network_file_prefix)]
        xlsx_files = [f for f in xlsx_files if re.match(network_file_prefix, f)]
    else:
        network_file_prefix = ''

    print('For:', network_retults_path, network_file_prefix, 'found:', len(xlsx_files), ' files')

    epochs = []
    times  = []
    for file in xlsx_files:
        file_path    =  os.path.join(network_retults_path, file) 
        val_sheet    = pd.read_excel(file_path, sheet_name='val_history')
        last_epoch   = int(val_sheet['epoch'].iloc[-1])

        # convert to timedelta
        elapsed_time_str = val_sheet['elapsed_time'].iloc[-1]
        elapsed_time = pd.to_timedelta(elapsed_time_str)       
        
        epochs.append(last_epoch)
        times.append(elapsed_time)

    total_time = sum(times, timedelta())
    mean_time  = total_time / len(times)

    return sum(epochs) / len(epochs), epochs, mean_time

# this function calculate the training epochs mean of specific network across all datasets
# the network_path must be the same in all dataset_paths
def calculate_training_epochs_mean_per_network(network_path, dataset_paths):

    network_file_prefix = None
    if type(network_path) is (tuple or list):
        network_file_prefix = network_path[1]
        network_path        = network_path[0]

    means  = []
    epochs = []
    times  = []
    for dataset_path in dataset_paths:
        network_retults_path =  os.path.join(dataset_path, network_path) 
        mean, epoch, mean_time = calculate_training_epochs_mean_per_dataset(network_retults_path, network_file_prefix)
        means.append(mean)
        times.append(mean_time)
        epochs += epoch

    total_time = sum(times, timedelta())
    mean_time   = total_time / len(times)

    return sum(means) / len(means), epochs, mean_time


In [2]:

networks = ['DeepLabV3', 
            'ULite', 
            'UNext',
            # All versions of umergenet were saved in the same directory; we will use regular expressions to identify each one separately.
            ('UMergeNet', r'^UMergeNet-axial-\d-'),    
            ('UMergeNet', r'^UMergeNet-axial-dw-\d-'), 
            ('UMergeNet', r'^UMergeNet-atrous-\d-'), 
            ('UMergeNet', r'^UMergeNet-atrous-dw-\d-'), 
            ('UMergeNet', r'^UMergeNet-normal-\d-'), 
            ('UMergeNet', r'^UMergeNet-normal-dw-\d-'),

            # For UNets we don't need regular expressions
            ('colab_unets','UNet'), 
            ('colab_unets','AttU_Net'), 
            ('colab_unets','R2AttU_Net')]

datasets = ['../fuseg']

print("FUSEG ONLY")
for network in networks:
    print('-'*10, network, '-'*10)
    mean, epochs, time = calculate_training_epochs_mean_per_network(network, datasets)
    print("Mean:", mean)
    print("Epochs:", epochs)
    print("Mean time:", time)
    print("")

FUSEG ONLY
---------- DeepLabV3 ----------
For: ../fuseg/DeepLabV3  found: 3  files
Mean: 68.33333333333333
Epochs: [83, 71, 51]
Mean time: 0 days 00:20:08

---------- ULite ----------
For: ../fuseg/ULite  found: 3  files
Mean: 47.333333333333336
Epochs: [53, 40, 49]
Mean time: 0 days 00:17:05

---------- UNext ----------
For: ../fuseg/UNext  found: 3  files
Mean: 38.333333333333336
Epochs: [41, 44, 30]
Mean time: 0 days 00:11:03.666666666

---------- ('UMergeNet', '^UMergeNet-axial-\\d-') ----------
For: ../fuseg/UMergeNet ^UMergeNet-axial-\d- found: 3  files
Mean: 51.333333333333336
Epochs: [68, 51, 35]
Mean time: 0 days 00:20:52.666666666

---------- ('UMergeNet', '^UMergeNet-axial-dw-\\d-') ----------
For: ../fuseg/UMergeNet ^UMergeNet-axial-dw-\d- found: 3  files
Mean: 62.666666666666664
Epochs: [40, 85, 63]
Mean time: 0 days 00:20:57.666666666

---------- ('UMergeNet', '^UMergeNet-atrous-\\d-') ----------
For: ../fuseg/UMergeNet ^UMergeNet-atrous-\d- found: 3  files
Mean: 48.3333

In [3]:

networks = ['DeepLabV3', 
            'ULite', 
            'UNext',
            # All versions of umergenet were saved in the same directory; we will use regular expressions to identify each one separately.
            ('UMergeNet', r'^UMergeNet-axial-\d-'),    
            ('UMergeNet', r'^UMergeNet-axial-dw-\d-'), 
            ('UMergeNet', r'^UMergeNet-atrous-\d-'), 
            ('UMergeNet', r'^UMergeNet-atrous-dw-\d-'), 
            ('UMergeNet', r'^UMergeNet-normal-\d-'), 
            ('UMergeNet', r'^UMergeNet-normal-dw-\d-'),

            # For UNets we don't need regular expressions
            ('colab_unets','UNet'), 
            ('colab_unets','AttU_Net'), 
            ('colab_unets','R2AttU_Net')]
datasets = ['../fuseg','../glas','../isic2018', '../lars']

for network in networks:
    print('-'*10, network, '-'*10)
    mean, epochs, time = calculate_training_epochs_mean_per_network(network, datasets)
    print("Mean:", mean)
    print("Epochs:", epochs)
    print("Mean time:", time)
    print("")

---------- DeepLabV3 ----------
For: ../fuseg/DeepLabV3  found: 3  files
For: ../glas/DeepLabV3  found: 3  files
For: ../isic2018/DeepLabV3  found: 3  files
For: ../lars/DeepLabV3  found: 3  files
Mean: 55.41666666666667
Epochs: [83, 71, 51, 39, 30, 40, 36, 30, 34, 88, 68, 95]
Mean time: 0 days 00:15:22.583333333

---------- ULite ----------
For: ../fuseg/ULite  found: 3  files
For: ../glas/ULite  found: 3  files
For: ../isic2018/ULite  found: 3  files
For: ../lars/ULite  found: 3  files
Mean: 43.916666666666664
Epochs: [53, 40, 49, 37, 31, 48, 34, 33, 35, 52, 62, 53]
Mean time: 0 days 00:12:28.999999999

---------- UNext ----------
For: ../fuseg/UNext  found: 3  files
For: ../glas/UNext  found: 3  files
For: ../isic2018/UNext  found: 3  files
For: ../lars/UNext  found: 3  files
Mean: 44.833333333333336
Epochs: [41, 44, 30, 37, 41, 38, 49, 32, 49, 63, 45, 69]
Mean time: 0 days 00:10:12.249999999

---------- ('UMergeNet', '^UMergeNet-axial-\\d-') ----------
For: ../fuseg/UMergeNet ^UMer

In [4]:

# like calculate_training_epochs_mean_per_dataset but for yolo
def calculate_training_epochs_mean_per_dataset_yolo(network_retults_path):

    yolo_trains =  os.listdir(network_retults_path)

    for train_path in yolo_trains:

        yolo_result_path = os.path.join(network_retults_path, train_path)        
        csv_files = [f for f in os.listdir(yolo_result_path) if f.endswith(".csv")]

        epochs = []
        times  = []
        for file in csv_files:
            file_path    = os.path.join(yolo_result_path, file) 
            val_sheet    = pd.read_csv(file_path)
            last_epoch   = int(val_sheet['epoch'].iloc[-1])
            elapsed_seconds = float(val_sheet['time'].iloc[-1])
            elapsed_time = pd.to_timedelta(elapsed_seconds, unit='s')

            epochs.append(last_epoch)
            times.append(elapsed_time)

    total_time = sum(times, pd.Timedelta(0))
    mean_time  = total_time / len(times)

    return sum(epochs) / len(epochs), epochs, mean_time

#print(calculate_training_epochs_mean_per_dataset_yolo('glas/yolo11'))


# like calculate_training_epochs_mean_per_network but for yolo
def calculate_training_epochs_mean_per_network_yolo(network_path, dataset_paths):
    means  = []
    epochs = []
    times  = []
    for dataset_path in dataset_paths:
        network_retults_path =  os.path.join(dataset_path, network_path) 
        mean, epoch, mean_time = calculate_training_epochs_mean_per_dataset_yolo(network_retults_path)
        means.append(mean)
        times.append(mean_time)
        epochs += epoch

    total_time = sum(times, timedelta())
    mean_time   = total_time / len(times)

    return sum(means) / len(means), epochs, mean_time





In [5]:
datasets = ['../fuseg']


print('-'*10, 'yolo11', '-'*10)
mean, epochs, time = calculate_training_epochs_mean_per_network_yolo('yolo11', datasets)
print("FUSEG ONLY")
print("Mean:", mean)
print("Epochs:", epochs)
print("Mean time:", time)

---------- yolo11 ----------
FUSEG ONLY
Mean: 61.0
Epochs: [61]
Mean time: 0 days 00:47:38.150000


In [6]:
datasets = ['../fuseg','../glas','../isic2018', '../lars']


print('-'*10, 'yolo11', '-'*10)
mean, epochs, time = calculate_training_epochs_mean_per_network_yolo('yolo11', datasets)
print("Mean:", mean)
print("Epochs:", epochs)
print("Mean time:", time)

---------- yolo11 ----------
Mean: 87.25
Epochs: [61, 142, 34, 112]
Mean time: 0 days 00:29:23.155750
